In [11]:
import os
import pandas as pd
from supabase import create_client, Client

# 1. Base URL wrapped in quotes (removed /rest/v1/)
SUPABASE_URL = "https://pimvvvtduabnhawaprle.supabase.co"

# 2. Key wrapped in quotes
SUPABASE_KEY = "sb_publishable_hZUCSSpPDUPLbRdwkVnhgQ_p9TPf2O_"

# 3. Initialize Supabase client using your defined variables
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)



In [12]:
# Fetch sales data
response_sales = supabase.table('sales').select('*').execute()
df_sales = pd.DataFrame(response_sales.data)

# Check shape and view first few rows
print("Sales Table Shape:", df_sales.shape)
df_sales.head()

Sales Table Shape: (1000, 12)


,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method
0,1,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart
1,2,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks
2,3,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks
3,4,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha
4,5,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart


In [13]:
# Fetch customers data
response_customers = supabase.table('customers').select('*').execute()
df_customers = pd.DataFrame(response_customers.data)

# Check shape and view first few rows
print("Customers Table Shape:", df_customers.shape)
df_customers.head()

Customers Table Shape: (1000, 9)


,customer_id,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,2637,Reet,Nurk,NaN,+372 5354 8280,Tallinn,2022-12-09,gold,1976
1,2723,Reet,Kuusik,NaN,+372 5603 8700,Tartu,2023-04-09,NaN,1998
2,2784,Mart,Pihl,NaN,+372 5536 5657,Tallinn,2022-10-30,gold,1989
3,3404,Maie,Tammik,NaN,+372 5291 9215,Tallinn,2020-03-26,NaN,2000
4,4278,Nele,Orav,NaN,+372 8700 9137,Tallinn,2024-05-10,NaN,1992


In [14]:
# Merge sales and customers
df = pd.merge(
    df_sales, 
    df_customers[['customer_id', 'email', 'first_name']], 
    on='customer_id', 
    how='left'
)

# Print verification details for Role A Output
print("Merged DataFrame Shape:", df.shape)
print("\nData Types:")
print(df.dtypes)
print("\nMerged DataFrame Preview:")
df.head()

Merged DataFrame Shape: (1000, 14)

Data Types:
id                  int64
sale_id             int64
invoice_id            str
sale_date             str
customer_id       float64
product_id          int64
quantity            int64
unit_price        float64
total_price       float64
channel               str
store_location        str
payment_method        str
email                 str
first_name            str
dtype: object

Merged DataFrame Preview:


,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,email,first_name
0,1,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart,NaN,Hille
1,2,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks,NaN,NaN
2,3,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks,NaN,NaN
3,4,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha,NaN,NaN
4,5,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart,triin.lill@telia.ee,Triin


In [15]:
print(f"Role B Starting Row Count: {df.shape[0]:,} rows")

Role B Starting Row Count: 1,000 rows


In [16]:
# Print duplicates before dropping
duplicates_found = df.duplicated(subset=['invoice_id']).sum()
print(f"Duplicates found before cleaning: {duplicates_found:,}")

# Drop duplicate invoices
df = df.drop_duplicates(subset=['invoice_id'], keep='first').copy()
print(f"Rows remaining after deduplication: {df.shape[0]:,}")

Duplicates found before cleaning: 0
Rows remaining after deduplication: 1,000


In [17]:
# Check missing values count
print("NULL values per column:")
print(df.isnull().sum())

# Drop rows missing critical identifiers or metrics
df = df.dropna(subset=['customer_id', 'sale_date', 'total_price']).copy()
print(f"\nRows remaining after dropping NULLs: {df.shape[0]:,}")

NULL values per column:
id                  0
sale_id             0
invoice_id          0
sale_date           0
customer_id        77
product_id          0
quantity            0
unit_price          0
total_price         0
channel             0
store_location    312
payment_method      0
email             752
first_name        725
dtype: int64

Rows remaining after dropping NULLs: 923


In [18]:
# Convert sale_date column to datetime
df['sale_date'] = pd.to_datetime(df['sale_date'])

# Verify data type change
print(f"sale_date dtype: {df['sale_date'].dtype}")

sale_date dtype: datetime64[us]


In [19]:
# Check count of negative or zero values
invalid_prices = (df['total_price'] <= 0).sum()
print(f"Invalid total_price rows (<= 0): {invalid_prices:,}")

# Keep only positive prices
df = df[df['total_price'] > 0].copy()
print(f"Rows remaining after filtering outliers: {df.shape[0]:,}")

Invalid total_price rows (<= 0): 16
Rows remaining after filtering outliers: 907


In [20]:
print("=" * 45)
print("             CLEANING REPORT             ")
print("=" * 45)
print(f"Final Cleaned DataFrame Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Total Unique Customers:       {df['customer_id'].nunique():,}")
print(f"Date Range Start (Min):       {df['sale_date'].min()}")
print(f"Date Range End   (Max):       {df['sale_date'].max()}")
print("=" * 45)

             CLEANING REPORT             
Final Cleaned DataFrame Shape: 907 rows, 14 columns
Total Unique Customers:       661
Date Range Start (Min):       2023-01-01 00:00:00
Date Range End   (Max):       2026-03-17 00:00:00
